# 📈 View — Receita Mensal

Validação da view `vw_receita_mensal` antes de mover para o Streamlit.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
pagamentos = pd.read_csv("../dados/pagamentos_limpo.csv")

print("Dados carregados!")

Dados carregados!


In [5]:
pedidos.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


In [4]:
pagamentos.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## 🧪 Testando o código antes de criar a view

In [2]:
# Join entre pedidos e pagamentos
df = pedidos.merge(pagamentos, on='order_id', how='left')

# Filtrando só pedidos entregues

df= df[df['order_status']== 'delivered']

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.00,credit_card,1.00,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,3.00,voucher,1.00,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2.00,voucher,1.00,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.00,boleto,1.00,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.00,credit_card,3.00,179.12


In [3]:
# Extraindo ano e mês

df['ano'] = df['order_purchase_timestamp'].dt.year
df['mes'] = df['order_purchase_timestamp'].dt.month
df['mes_nome'] = df['order_purchase_timestamp'].dt.strftime('%b')
df['ano_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

#agrupando mês

receita_mensal = (df.groupby(['ano', 'mes', 'mes_nome', 'ano_mes'])
                    .agg(
                        total_pedidos = ('order_id', 'nunique'),
                        receita_total = ('payment_value', 'sum')
                    )
                    .reset_index() #O reset_index() transforma de volta em colunas normais 
                    .sort_values(['ano', 'mes'])) #Ordena por ano e mês em ordem crescente

receita_mensal['receita_total'] = receita_mensal['receita_total'].round(2) #Arredonda a receita para 2 casas decimais.

receita_mensal.head()

,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total
0,2016,9,Sep,2016-09,1,0.00
1,2016,10,Oct,2016-10,265,46566.71
2,2016,12,Dec,2016-12,1,19.62
3,2017,1,Jan,2017-01,750,127545.67
4,2017,2,Feb,2017-02,1653,271298.65


In [4]:
#tradução mes_nome

meses_traducao = {
    'Jan': 'Jan', 'Feb': 'Fev', 'Mar': 'Mar',
    'Apr': 'Abr', 'May': 'Mai', 'Jun': 'Jun',
    'Jul': 'Jul', 'Aug': 'Ago', 'Sep': 'Set',
    'Oct': 'Out', 'Nov': 'Nov', 'Dec': 'Dez'
}

receita_mensal['mes_nome'] = receita_mensal['mes_nome'].map(meses_traducao)

receita_mensal.head()

,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total
0,2016,9,Set,2016-09,1,0.00
1,2016,10,Out,2016-10,265,46566.71
2,2016,12,Dez,2016-12,1,19.62
3,2017,1,Jan,2017-01,750,127545.67
4,2017,2,Fev,2017-02,1653,271298.65


In [5]:
# LAG — pega o valor do mês anterior dentro do mesmo ano
# equivale ao LAG() do SQL

receita_mensal['receita_mes_anterior'] = (
    receita_mensal.groupby('ano')['receita_total']
    .shift(1) #Desloca os valores uma posição para baixo dentro do grupo — equivale ao LAG() do BigQuery 
)

# Calculando variação MoM
receita_mensal['variacao_mom_pct'] = (
    (receita_mensal['receita_total'] - receita_mensal['receita_mes_anterior'])
    / receita_mensal['receita_mes_anterior'] * 100
).round(2)

receita_mensal.head(15)

,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total,receita_mes_anterior,variacao_mom_pct
0,2016,9,Set,2016-09,1,0.00,NaN,NaN
1,2016,10,Out,2016-10,265,46566.71,0.00,inf
2,2016,12,Dez,2016-12,1,19.62,46566.71,-99.96
3,2017,1,Jan,2017-01,750,127545.67,NaN,NaN
4,2017,2,Fev,2017-02,1653,271298.65,127545.67,112.71
5,2017,3,Mar,2017-03,2546,414369.39,271298.65,52.74
6,2017,4,Abr,2017-04,2303,390952.18,414369.39,-5.65
7,2017,5,Mai,2017-05,3546,567066.73,390952.18,45.05
8,2017,6,Jun,2017-06,3135,490225.60,567066.73,-13.55
9,2017,7,Jul,2017-07,3872,566403.93,490225.60,15.54


In [6]:
receita_mensal['variacao_mom_pct'] = receita_mensal['variacao_mom_pct'].replace([np.inf, -np.inf], np.nan)

receita_mensal.head()

,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total,receita_mes_anterior,variacao_mom_pct
0,2016,9,Set,2016-09,1,0.00,NaN,NaN
1,2016,10,Out,2016-10,265,46566.71,0.00,NaN
2,2016,12,Dez,2016-12,1,19.62,46566.71,-99.96
3,2017,1,Jan,2017-01,750,127545.67,NaN,NaN
4,2017,2,Fev,2017-02,1653,271298.65,127545.67,112.71


In [7]:
from views.vw_receita_mensal import get_receita_mensal

df_receita = get_receita_mensal(pedidos, pagamentos)
df_receita.head()

,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total,receita_mes_anterior,variacao_mom_pct
0,2016,9,Set,2016-09,1,0.00,NaN,NaN
1,2016,10,Out,2016-10,265,46566.71,0.00,NaN
2,2016,12,Dez,2016-12,1,19.62,46566.71,-99.96
3,2017,1,Jan,2017-01,750,127545.67,NaN,NaN
4,2017,2,Fev,2017-02,1653,271298.65,127545.67,112.71
